# Perceptron clássico e classificador logístico em PyTorch

Este notebook separa explicitamente **dois classificadores lineares relacionados, mas diferentes**:

1. o **Perceptron clássico de Rosenblatt**, com ativação degrau e atualização baseada em erros de classificação;
2. o **classificador logístico** (uma unidade linear com sigmóide), treinado por minimização de entropia cruzada com gradientes.

Ambos usam a mesma transformação afim

$$z = \mathbf{w}^T\mathbf{x} + b,$$

mas **não usam a mesma regra de treinamento**.

> `nn.Linear + Sigmoid/BCE` implementa regressão logística binária, não o algoritmo clássico do Perceptron.


## 1. Transformação afim e fronteira de decisão

A analogia com sinapses pode ser útil como recurso mnemônico, mas o modelo matemático abaixo é um **classificador linear**, não um modelo fisiologicamente plausível de um neurônio.

Para uma entrada $\mathbf{x}\in\mathbb{R}^d$, pesos $\mathbf{w}\in\mathbb{R}^d$ e intercepto $b\in\mathbb{R}$, definimos

$$z = \mathbf{w}^T\mathbf{x}+b.$$

O valor $z$ é o resultado de uma **transformação afim**. A equação

$$\mathbf{w}^T\mathbf{x}+b=0$$

define um hiperplano: a fronteira que separa os semiespaços $z<0$ e $z>0$.

O intercepto $b$ desloca essa fronteira. Um limiar diferente de zero pode sempre ser absorvido no intercepto.


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# Configurações de visualização
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['axes.edgecolor'] = '#30363d'
plt.rcParams['text.color'] = '#e6edf3'
plt.rcParams['axes.labelcolor'] = '#e6edf3'
plt.rcParams['xtick.color'] = '#8b949e'
plt.rcParams['ytick.color'] = '#8b949e'
plt.rcParams['grid.color'] = '#21262d'

torch.manual_seed(42)

# AND lógico: conjunto linearmente separável
X = torch.tensor([[0., 0.],
                  [0., 1.],
                  [1., 0.],
                  [1., 1.]])

y01 = torch.tensor([[0.], [0.], [0.], [1.]])
y_pm = 2 * y01.flatten() - 1  # rótulos {-1, +1} para o Perceptron clássico

print('Dados do AND:')
for xi, yi in zip(X, y01):
    print(f'x={xi.tolist()}  y={int(yi.item())}')


## 2. Da transformação afim à decisão: degrau versus sigmóide

O **Perceptron clássico** usa uma decisão discreta:

$$\hat y_{\text{perc}} = \begin{cases} 1, & \text{if } z \ge 0 \\ 0, & \text{c.c.} \end{cases}$$

Já a regressão logística usa a função sigmóide

$$\sigma(z)=\frac{1}{1+e^{-z}},$$

interpretando $\sigma(z)$ como $P(y=1\mid\mathbf{x})$ sob o modelo logístico.

Se classificarmos a saída logística com limiar $0.5$, então

$$\sigma(z)\ge 0.5 \iff z\ge 0.$$

Portanto, **os dois modelos podem induzir a mesma fronteira de decisão linear**, embora suas saídas e regras de treinamento sejam diferentes.

A função degrau não fornece um gradiente útil para treinamento por retropropagação: sua derivada é zero quase em toda parte e não é definida no limiar. A sigmóide é diferenciável.


In [ ]:
z_range = torch.linspace(-6, 6, 300)
step = (z_range >= 0).float()
sigmoid = torch.sigmoid(z_range)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(z_range.numpy(), step.numpy(), lw=2.5)
axes[0].axvline(0, linestyle='--', alpha=0.7)
axes[0].set_title('Degrau: Perceptron clássico')
axes[0].set_xlabel('z')
axes[0].set_ylabel('saída')
axes[0].grid(True, alpha=0.3)

axes[1].plot(z_range.numpy(), sigmoid.numpy(), lw=2.5)
axes[1].axvline(0, linestyle='--', alpha=0.7)
axes[1].axhline(0.5, linestyle='--', alpha=0.7)
axes[1].set_title('Sigmoid: regressão logística')
axes[1].set_xlabel('z')
axes[1].set_ylabel('σ(z)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Algoritmo clássico do Perceptron

Usaremos os rótulos $y_i\in\{0, 1\}$. Para uma amostra $(\mathbf{x}_i,y_i)$, definimos o score (logit) como

$$z_i = \mathbf{w}^T\mathbf{x}_i+b,$$

e a predição usando a função degrau (step):

$$\hat{y}_i = \begin{cases} 1 & \text{se } z_i > 0 \\ 0 & \text{caso contrário} \end{cases}$$

Se a predição estiver incorreta (ou seja, $\hat{y}_i \neq y_i$), a amostra está errada. A atualização é baseada no erro:

$$\mathbf{w}\leftarrow\mathbf{w}+\eta(y_i - \hat{y}_i)\mathbf{x}_i,$$

$$b\leftarrow b+\eta(y_i - \hat{y}_i),$$

onde $\eta>0$ é a taxa de aprendizado.

**Por que a diferença funciona com {0, 1}?**
- Se $y=1$ e $\hat{y}=0$, o erro é $+1$. Somamos o vetor $\mathbf{x}_i$ aos pesos para empurrar a predição para cima.
- Se $y=0$ e $\hat{y}=1$, o erro é $-1$. Subtraímos o vetor $\mathbf{x}_i$ dos pesos para empurrar a predição para baixo.
- Se a predição está certa, erro é $0$ e nada muda.

**Diferença para o Perceptron Original (1958)**
O algoritmo original proposto por Frank Rosenblatt era matematicamente idêntico no resultado, mas usava classes $\{-1, +1\}$. Na versão original, verificava-se a "signed margin" (se $y_i z_i \le 0$) e, em caso de erro, multiplicava-se a atualização apenas por $y_i$ (sem calcular a diferença $y_i - \hat{y}_i$).
Nós adotamos a formulação com classes $\{0, 1\}$ e a regra da diferença $(y_i - \hat{y}_i)$ (conhecida como regra delta, que surgiu com o Adaline em 1960) porque:
1. É a notação padrão usada hoje na literatura de Redes Neurais (onde as classes começam em 0);
2. A regra baseada em diferença do erro transiciona naturalmente para a Regressão Logística e para o Backpropagation, onde a magnitude do erro dita o tamanho do passo.

Essa regra não precisa de `loss.backward()` nem de uma ativação diferenciável.

### Teorema de convergência do Perceptron

Se os dados forem linearmente separáveis, o algoritmo com taxa positiva fixa comete um número finito de erros. Se os dados não forem linearmente separáveis, essa garantia desaparece e as atualizações podem continuar indefinidamente.

In [ ]:
def treinar_perceptron_classico(X_, y_, eta=1.0, max_epocas=50):
    w = torch.zeros(X_.shape[1])
    b = torch.tensor(0.0)
    erros_por_epoca = []

    for _ in range(max_epocas):
        erros = 0
        for xi, yi in zip(X_, y_):
            z = torch.dot(w, xi) + b
            # Função degrau para classes 0 e 1
            y_pred = torch.tensor(1.0) if z > 0 else torch.tensor(0.0)
            erro = yi - y_pred
            if erro != 0:
                w = w + eta * erro * xi
                b = b + eta * erro
                erros += 1

        erros_por_epoca.append(erros)
        if erros == 0:
            break

    return w, b, erros_por_epoca

# Passamos y01 (que contém as classes 0 e 1) no formato apropriado
w_perc, b_perc, erros_perc = treinar_perceptron_classico(X, y01.flatten())

print('Perceptron treinado com regra de erro (classes 0 e 1):')
print(f'w = {w_perc.tolist()}')
print(f'b = {b_perc.item():.3f}')
print(f'erros por época = {erros_perc}')

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))

for xi, yi in zip(X.numpy(), y01.numpy().flatten()):
    c = 'C0' if yi == 0 else 'C1'
    ax.scatter(xi[0], xi[1], s=180, c=c, edgecolors='white', linewidths=1.5,
               label=f'classe {int(yi)}' if int(yi) not in [int(t.get_label().split()[-1]) for t in ax.collections if t.get_label().startswith('classe')] else None)

x_line = np.linspace(-0.5, 1.5, 200)
if abs(w_perc[1].item()) > 1e-12:
    y_line = -(w_perc[0].item() * x_line + b_perc.item()) / w_perc[1].item()
    ax.plot(x_line, y_line, '--', lw=2, label=r'$w^Tx+b=0$')

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title('Fronteira aprendida pelo Perceptron clássico')
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## 4. Alternativa suave: regressão logística

Agora trocamos deliberadamente de algoritmo.

Definimos

$$p_i=\sigma(z_i),\qquad z_i=\mathbf{w}^T\mathbf{x}_i+b,$$

com a entropia cruzada binária (BCE)

$$L=-\frac1n\sum_i \left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].$$

Para uma única amostra, a derivada em relação ao logit $z$ é especialmente simples:

$$\frac{\partial \ell}{\partial z}=\sigma(z)-y.$$

Logo,

$$\nabla_{\mathbf w}\ell=(\sigma(z)-y)\mathbf{x},\qquad
\frac{\partial \ell}{\partial b}=\sigma(z)-y.$$

É **aqui** que o gradiente entra: adotamos uma função de erro suave e diferenciável para um modelo logístico.

Em PyTorch, `nn.BCEWithLogitsLoss()` é preferível a aplicar `Sigmoid` e depois `BCELoss`, porque combina as duas operações de forma numericamente mais estável. O modelo abaixo retorna **logits**, não probabilidades.


In [ ]:
class ClassificadorLogistico(nn.Module):
    def __init__(self, n_entradas):
        super().__init__()
        self.linear = nn.Linear(n_entradas, 1)

    def forward(self, x_):
        return self.linear(x_)  # logits z = w^T x + b


torch.manual_seed(0)
modelo = ClassificadorLogistico(n_entradas=2)
criterio = nn.BCEWithLogitsLoss()
lambda_l2 = 0.05

# O otimizador chama-se SGD, mas como usamos o conjunto inteiro em cada passo,
# o procedimento abaixo é gradiente em batch completo, não SGD estocástico.
otimizador = torch.optim.SGD(modelo.parameters(), lr=0.5)

historico_objetivo = []
historico_bce = []
n_epocas = 200

for _ in range(n_epocas):
    logits = modelo(X)
    bce = criterio(logits, y01)
    reg = 0.5 * lambda_l2 * modelo.linear.weight.pow(2).sum()
    objetivo = bce + reg

    otimizador.zero_grad()
    objetivo.backward()
    otimizador.step()

    historico_bce.append(bce.item())
    historico_objetivo.append(objetivo.item())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(historico_bce, label='BCE')
ax.plot(historico_objetivo, label='BCE + regularização L2')
ax.set_xlabel('Época')
ax.set_ylabel('Valor')
ax.set_title('Treinamento do classificador logístico')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## Referências conceituais

- Rosenblatt, F. (1958). *The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain*.
- Novikoff, A. B. J. (1962). *On Convergence Proofs on Perceptrons*.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning* — classificação linear e regressão logística.
- PyTorch: `nn.Linear`, `nn.BCEWithLogitsLoss`, `torch.optim.SGD` e `torch.optim.Adam`.

---
*Versão revisada para separar explicitamente Perceptron clássico, regressão logística e otimização convexa.*
